In [ ]:
from pathlib import Path
import os
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "cifar10"]
_dataset_root = next(p for p in _candidates if (p / "baseline_cifar10.py").exists())
os.chdir(_dataset_root)


In [ ]:
#importing the required libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time

In [ ]:
# loading CIFAR-10 dataset
# dataset has images of handwritten digits from 0 to 9
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0

# change images from 32x32x3 to 3072
x_train = x_train.reshape(-1, 3072)
x_test = x_test.reshape(-1, 3072)

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

In [ ]:
# Build a simple baseline model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(3072,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Train the model

start_time = time.time()

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

training_time = time.time() - start_time
print("Training time:", round(training_time, 2), "seconds")

In [ ]:
# Test baseline model

test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Baseline Model Accuracy:", round(test_accuracy * 100, 2), "%")

In [ ]:
# Save baseline model

model.save("baseline_cifar10_model.h5")

baseline_size = os.path.getsize("baseline_cifar10_model.h5") / 1024
print("Baseline Model Size:", round(baseline_size, 2), "KB")

In [ ]:
# Convert baseline model to TensorFlow Lite

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("baseline_model.tflite", "wb") as f:
    f.write(tflite_model)

tflite_size = os.path.getsize("baseline_model.tflite") / 1024
print("TFLite Model Size:", round(tflite_size, 2), "KB")

In [ ]:
# Apply quantization

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_model = converter.convert()

with open("quantized_model.tflite", "wb") as f:
    f.write(quantized_model)

quantized_size = os.path.getsize("quantized_model.tflite") / 1024
print("Quantized Model Size:", round(quantized_size, 2), "KB")

In [ ]:
# Compare model sizes

models = ["Baseline H5", "TFLite", "Quantized TFLite"]
sizes = [baseline_size, tflite_size, quantized_size]

plt.figure(figsize=(8,5))
plt.bar(models, sizes)
plt.ylabel("Model Size (KB)")
plt.title("Model Size Comparison")
plt.show()

In [ ]:

import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["Baseline H5", "TFLite", "Quantized TFLite"],
    "Model Size (KB)": [baseline_size, tflite_size, quantized_size],
})

comparison_df

In [ ]:
accuracy_df = pd.DataFrame({
    "Metric": ["Baseline Accuracy", "Training Time (sec)", "Size Reduction (%)"],
    "Value": [
        round(test_accuracy * 100, 2),
        round(training_time, 2),
        round(reduction, 2)
    ]
})

accuracy_df